# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

Hugging Face READ token (hf_...): ··········
connected; seed = 42


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Two findings from *The State of AI-Driven SEO, March 2026* (`docs/flyrank-seo-research-march-2026.pdf`). The paper is careful in a lot of places (it demotes its ML to an appendix and flags small buckets), so these are questions about how to make two claims stronger, not gotchas.

**Finding #4: "365+ day content refreshed within 30 days shows a 3.2× health boost and 57× more impressions."**
- *Where does the label come from?* The outcome is the health score and 90-day impressions in the same snapshot in which freshness is measured. The health score is itself built from impressions, position, CTR and scroll, so "health" and "impressions" aren't two independent outcomes.
- *Does the design carry the claim?* It compares pages that *were* refreshed with pages that weren't. Pages get chosen for refresh, often because they matter or because someone noticed them, so part of the gap is the choosing, not the refreshing. The impression window also partly comes before the refresh date. The playbook line "refreshing mature pages produces 3.2× health" is causal wording on an observational comparison.
- *How to make it stronger:* use the daily fact to compare each refreshed page's impressions in a fixed window before vs after its update date, against matched pages from the same site that weren't touched (a difference-in-differences), grouped by brand.

**ML appendix: "Logistic regression (71% holdout accuracy) separates growing from declining pages."**
- *Where does the label come from?* Growth/decline is the 30-day vs previous-30-day impression change, the same window that features like recent impressions and days visible are computed from. So the features overlap the label period.
- *Does the design carry the claim?* It's an 80/20 random split across 57 brands, so pages from the same brand sit on both sides. And 71% is printed without its base rate: with 74,187 growing vs 45,272 declining pages, always guessing "growing" already scores 62%. The cell below does that arithmetic.
- *How to make it stronger:* report the majority-class baseline and AUC, split by brand (GroupKFold), and define growth on a window strictly after the features. That's what my own setup does.

In [2]:
# Put the paper's headline ML number next to the base rate it did not print.
rising, declining = 74_187, 45_272          # Finding #1 cohort sizes (growing vs declining pages)
majority = max(rising, declining) / (rising + declining)
print(f'growth-vs-decline logistic regression: 71% holdout accuracy; always-guess-"growing" accuracy = {majority:.3f}')
print(f'-> skill over the majority class: {0.71 - majority:+.3f}, measured on a random 80/20 split (not grouped by brand)')

growth-vs-decline logistic regression: 71% holdout accuracy; always-guess-"growing" accuracy = 0.621
-> skill over the majority class: +0.089, measured on a random 80/20 split (not grouped by brand)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

June 2026 is opened here for the first time. The recipe (gradient boosting, 18 features) was frozen in w05 and isn't touched again.

**After (honest):** each client is scored by a model trained only on *other* clients' March and April snapshots, and judged on June outcomes. That's 26 clients with 30+ pages each. Gradient boosting reaches **ROC-AUC 0.704** against **0.602** for the rule, and average precision 0.715 vs 0.605. Inside each site, the model's top-10 were right **0.89** of the time, against **0.75** for the rule and **0.52** for random picking. Logistic regression, the runner-up from w05, lands in the same place (AUC 0.701, per-site top-10 0.91), so the result doesn't hinge on which model I froze.

**Before (flattering):** one random 75/25 row split over all snapshots and clients gives ROC-AUC 0.798. The gap between that and the honest 0.704 is the part that came from memorising clients and months. It's the same lesson as notebook 03, now measured properly.

The sealed-frame builder (this cell) and its metrics file (`work/outputs/sealed_metrics.json`) are both committed, so "evaluated once on June" can be checked from the repo.

In [3]:
# Build the per-item panel for all decision dates incl. the sealed one and the deploy date (one scan).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.ALL_DATES + [ff.DEPLOY_DATE])
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
lab = m[m.decision_date != ff.DEPLOY_DATE].copy()      # rows with an observed outcome month
dep = m[m.decision_date == ff.DEPLOY_DATE].copy()      # today's queue: the label month is the real future
print(f'{len(panel):,} item-date rows in {time.time()-t0:.0f}s; labelled rows {len(lab):,}; deploy rows {len(dep):,}')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
MAKERS = {
    'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'hgb': lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                                  min_samples_leaf=100, random_state=ff.SEED),
    'rf': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=ff.SEED),
}
FINAL = 'hgb'    # frozen in w05 on the development split (best top-of-queue precision), never re-chosen later
F = ff.FEATURES

def baseline(df):
    """Frozen transparent rule: one point per warning sign, ties broken by current visibility."""
    r = pd.DataFrame(index=df.index)
    r['sliding_impressions'] = df.imp_last30 < 0.85 * df.imp_prev30      # already losing impressions
    r['position_slipping']   = df.pos_change > 1.0                        # average rank got >1 place worse
    r['one_day_spike']       = df.spike_share > 0.25                      # month propped up by a single day
    r['aging_page']          = df.content_age_days > 270                  # past the decay cliff in FlyRank's paper
    r['patchy_visibility']   = df.days_with_imp_last30 < 15               # visible on fewer than half the days
    points = r.sum(axis=1)
    score = points + df.log_imp_last30 / 100.0
    reasons = r.apply(lambda row: '|'.join(c for c in r.columns if row[c]) or 'none', axis=1)
    return score, reasons, r

from sklearn.metrics import roc_auc_score, average_precision_score
def evaluate(scores, y, ks=(20, 50, 100, 500)):
    y = np.asarray(y).astype(int)
    d = {'base_rate': round(float(y.mean()), 4), 'auc': round(float(roc_auc_score(y, scores)), 4),
         'ap': round(float(average_precision_score(y, scores)), 4)}
    for k in ks:
        d[f'p@{k}'] = round(ff.precision_at_k(scores, y, k), 4)
    return d
def client_queue_at_10(df, scores, min_items=30):
    """How a single site would use it: precision of its OWN top-10, averaged over clients
    (with those clients' average base rate next to it)."""
    tmp = df[['client_hash_id', 'declined']].copy(); tmp['s'] = np.asarray(scores)
    rows = [(g.sort_values('s', ascending=False).declined.head(10).mean(), g.declined.mean())
            for _, g in tmp.groupby('client_hash_id') if len(g) >= min_items]
    return {'client_p@10': round(float(np.mean([a for a, b in rows])), 3),
            'client_base': round(float(np.mean([b for a, b in rows])), 3), 'clients': len(rows)}

T1, T2 = ff.DEV_DATES; T3 = ff.SEALED_DATE
# AFTER (honest): client-grouped + time-forward. Train on other clients' March/April snapshots, judge June outcomes.
test = ff.grouped_forward_scores(lab, [T1, T2], T3, MAKERS[FINAL], F)
b, _, _ = baseline(test)
sealed = {'baseline_rule': {**evaluate(b, test.declined), **client_queue_at_10(test, b)},
          FINAL: {**evaluate(test.score, test.declined), **client_queue_at_10(test, test.score)}}
lr_scores = ff.grouped_forward_scores(lab, [T1, T2], T3, MAKERS['logreg'], F)
sealed['logreg'] = {**evaluate(lr_scores.score, lr_scores.declined), **client_queue_at_10(lr_scores, lr_scores.score)}
# BEFORE (the version that flatters itself): one random row split over every snapshot and client.
pool = lab.sample(frac=1.0, random_state=ff.SEED); cut = int(len(pool) * 0.75)
rtr, rte = pool.iloc[:cut], pool.iloc[cut:]
p_rand = MAKERS[FINAL]().fit(rtr[F], rtr.declined.astype(int)).predict_proba(rte[F])[:, 1]
before_after = {'random row split (before)': evaluate(p_rand, rte.declined),
                'client-grouped + future month (after)': sealed[FINAL]}
print(f'sealed frame: {len(test):,} June-outcome rows from {test.client_hash_id.nunique()} clients, base rate {test.declined.mean():.3f}')
print(pd.DataFrame(before_after).T.to_string())
print('\nsealed test, all methods:'); print(pd.DataFrame(sealed).T.to_string())
receipt = {'frame': {'train_dates': [T1, T2], 'test_date': T3, 'label_month': '2026-06', 'n_test': int(len(test)),
                     'n_test_clients': int(test.client_hash_id.nunique()), 'folds': 4, 'seed': ff.SEED, 'model': FINAL},
           'sealed': sealed, 'before_after': before_after}
json.dump(receipt, open('work/outputs/sealed_metrics.json', 'w'), indent=2, default=str)
print('\nRECEIPT_JSON ' + json.dumps(receipt, default=str))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

433,772 item-date rows in 275s; labelled rows 219,935; deploy rows 87,664
sealed frame: 85,178 June-outcome rows from 37 clients, base rate 0.519
                                       base_rate     auc      ap  p@20  p@50  p@100  p@500  client_p@10  client_base  clients
random row split (before)                 0.5001  0.7977  0.7977   1.0  1.00   1.00  0.994          NaN          NaN      NaN
client-grouped + future month (after)     0.5186  0.7038  0.7151   1.0  0.94   0.97  0.964        0.885        0.517     26.0

sealed test, all methods:
               base_rate     auc      ap  p@20  p@50  p@100  p@500  client_p@10  client_base  clients
baseline_rule     0.5186  0.6016  0.6054   0.8  0.86   0.71  0.826        0.746        0.517     26.0
hgb               0.5186  0.7038  0.7151   1.0  0.94   0.97  0.964        0.885        0.517     26.0
logreg            0.5186  0.7010  0.7099   1.0  0.96   0.95  0.952        0.908        0.517     26.0

RECEIPT_JSON {"frame": {"train_dates": [

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The week-3 hunt again, on the final feature set and the sealed split:

- No feature is built from the outcome month, from export-time edit dates, from the query table, or from IDs.
- No client appears on both sides of any fold. That's enforced by construction in `grouped_forward_scores`.
- **Harness test:** adding the outcome month's impressions as a feature pushes AUC up sharply. So the setup *would* catch a leak, and my honest number isn't hiding one.
- **Population check:** which rows are kept depends only on (T-60, T] volume, page age at T and the client's feed start. No outcome information is used to choose rows.
- **Disclosed design choice:** the label compares each page with its own site in the outcome month. That uses June information *by definition*. It belongs to the label, not the features, and it's the reason site-wide shocks don't count as page declines.

**Real failure examples on the sealed month** (tables below the checks): the most confident false alarms are pages that had already crashed the month before (12,732 → 1,773 and 9,929 → 1,119 impressions, rank down 4-9 places) on a site whose total fell to about a quarter in June. They stopped falling while the site collapsed, so relative to the site they held. The most confident misses are the mirror image: big, steady page-one pages (≈6,700 and ≈8,600 impressions, position ≈4, nothing changing) on a site that *grew* in June. They stayed flat while the site rose, so they count as falling behind, and nothing on the page warned of it. Both come from the site-relative label meeting a site-wide swing the features can't see coming. By volume, ranking is weakest on the lower three quartiles (AUC 0.66-0.68) and best on the biggest pages (0.75). That's the reverse of the development month, so where the model struggles shifts from month to month.

In [4]:
from sklearn.metrics import roc_auc_score
checks = [('features computed from (T, T+30]', [f for f in F if 'next' in f or 'ratio' in f] or 'none'),
          ('export-time edit fields in F', [f for f in F if 'updated' in f or 'optimized' in f] or 'none'),
          ('query-table fields in F', [f for f in F if 'query' in f or 'rare' in f or 'anon' in f] or 'none'),
          ('ID columns in F', [f for f in F if 'hash' in f] or 'none')]
for k, v in checks:
    print(f'{k:40} {v}')
lab2 = lab.copy(); lab2['LEAK'] = np.log1p(lab2.imp_next30)
o_leak = ff.grouped_forward_scores(lab2, [T1, T2], T3, MAKERS[FINAL], F + ['LEAK'])
print(f"\nharness test: AUC with an outcome-month feature = {roc_auc_score(o_leak.declined.astype(int), o_leak.score):.3f} "
      f"(honest: {sealed[FINAL]['auc']:.3f}) -> the harness would catch a leak")
print('population filter uses only (T-60, T] volume, page age at T and the client feed start -> no outcome-window information')
print('but note: the LABEL compares each page to its own site in the outcome month - a design choice, disclosed')
# Real failure examples on the sealed month (only pre-T columns + the June outcome shown for review)
show = ['imp_prev30', 'imp_last30', 'imp_next30', 'client_ratio', 'pos_last30', 'pos_change', 'spike_share', 'content_age_days', 'score']
print('\nsealed June: 3 most confident false alarms (high score, held up):')
print(test[test.declined == 0].nlargest(3, 'score')[show].round(2).to_string(index=False))
print('\nsealed June: 3 most confident misses (low score, fell behind):')
print(test[test.declined == 1].nsmallest(3, 'score')[show].round(2).to_string(index=False))
q = pd.qcut(test.imp_last30.rank(method='first'), 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
print('\nwhere it is weakest - AUC inside each volume quartile:',
      {k: round(roc_auc_score(g.declined.astype(int), g.score), 3) for k, g in test.groupby(q, observed=True)})

features computed from (T, T+30]         none
export-time edit fields in F             none
query-table fields in F                  none
ID columns in F                          none

harness test: AUC with an outcome-month feature = 0.920 (honest: 0.704) -> the harness would catch a leak
population filter uses only (T-60, T] volume, page age at T and the client feed start -> no outcome-window information
but note: the LABEL compares each page to its own site in the outcome month - a design choice, disclosed

sealed June: 3 most confident false alarms (high score, held up):
 imp_prev30  imp_last30  imp_next30  client_ratio  pos_last30  pos_change  spike_share  content_age_days  score
    12732.0      1773.0      2059.0          0.27        5.97        4.47         0.17             215.0   0.99
     3493.0       373.0      1320.0          0.69       10.55        5.99         0.08             227.0   0.99
     9929.0      1119.0      1271.0          0.27       21.89        8.84         

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (from w01):** "the honest client-held-out split reaches 84.0% accuracy… a real, measured gap that shows why client-grouped validation matters."

What's wrong with it: the features overlapped the outcome window (pos_volatility over 180 days, query-table columns covering June), accuracy was printed without a base rate, and "honest" was doing more work than the design could carry.

**Rewritten:** *On a sealed month (June 2026) and on clients the model never saw, a gradient-boosting ranker built only from signals available on decision day ordered pages by their risk of falling more than 20% behind their own site with ROC-AUC 0.704, against 0.602 for a transparent five-sign rule. Inside each site its top-10 flags were right 0.89 of the time (rule: 0.75; random: 0.52). This is an observed, decision-support result on one pseudonymised portfolio. It says which pages look worth reviewing first, not that reviewing or refreshing them will change the outcome.*

In [5]:
s = receipt['sealed']
print(f"sealed June test, unseen clients: per-site top-10 precision {s[FINAL]['client_p@10']:.2f} (model) vs "
      f"{s['baseline_rule']['client_p@10']:.2f} (rule) vs {s[FINAL]['client_base']:.2f} (those sites' base rate); "
      f"ROC-AUC {s[FINAL]['auc']:.3f} vs {s['baseline_rule']['auc']:.3f}")

sealed June test, unseen clients: per-site top-10 precision 0.89 (model) vs 0.75 (rule) vs 0.52 (those sites' base rate); ROC-AUC 0.704 vs 0.602


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.